<a href="https://colab.research.google.com/github/Navyanagumothu3/E-Commerce-Sales-Analysis/blob/main/Fraud_Detection_Systemipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fraud Detection System**


Step 1: Load Dataset

Upload CSV to Colab and read with pandas.

In [17]:
import pandas as pd

df = pd.read_csv("creditcard.csv")
print(df.head())
print(df['Class'].value_counts())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

Step 2: Preprocess Data

Drop missing values, separate features and target, scale features, and split data.

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Drop rows with missing target (just in case)
df = df.dropna(subset=['Class'])

X = df.drop('Class', axis=1)
y = df['Class']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

Step 3: Train Model

Use a simple ML model for fraud detection.

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=500, random_state=42)

Step 4: Evaluate Model

Accuracy, ROC-AUC, and classification report.

In [22]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

Accuracy: 0.9991573329588147
ROC-AUC: 0.9599474004570878
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.64      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962



Step 5: Save Model and Scaler

Save trained objects for API use.

In [23]:
import joblib

joblib.dump(model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

 Flask App Initialization

In [25]:
!pip install pyngrok flask


Step 6: Flask API

Serve predictions via /predict endpoint.

In [27]:
from flask import Flask, request, jsonify
import numpy as np
import joblib

# Load model & scaler
model = joblib.load("fraud_model.pkl")
scaler = joblib.load("scaler.pkl")

app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json["data"]
    arr = np.array(data).reshape(1, -1)
    scaled = scaler.transform(arr)
    pred = model.predict(scaled)[0]
    proba = model.predict_proba(scaled)[0][1]
    return jsonify({"prediction": int(pred), "fraud_probability": float(proba)})

import threading
def run():
    app.run(host='0.0.0.0', port=5000)
threading.Thread(target=run).start()

 * Serving Flask app '__main__'
 * Debug mode: off


Step 7: Test API

Send a POST request with sample data.

In [28]:
import requests

data = {
    "data": [
        0, -1.359807134, -0.072781173, 2.536346738, 1.378155224,
        -0.33832077, 0.462387778, 0.239598554, 0.098697901,
        0.36378697, 0.090794172, -0.551599533, -0.617800856,
        -0.991389847, -0.311169354, 1.468176972, -0.470400525,
        0.207971242, 0.02579058, 0.40399296, 0.251412098,
        -0.018306778, 0.277837576, -0.11047391, 0.066928075,
        0.128539358, -0.189114844, 0.133558377, -0.021053053,
        149.62
    ]
}

response = requests.post("http://127.0.0.1:5000/predict", json=data)
print(response.json())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [19/Mar/2026 05:34:00] "POST /predict HTTP/1.1" 200 -


{'fraud_probability': 0.0005720866523626679, 'prediction': 0}


Evaluated using accuracy and ROC-AUC metrics

Step 8: Final Evaluation

Confirm model metrics with the saved model.

In [29]:
from sklearn.metrics import accuracy_score, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)

Accuracy: 0.9991573329588147
ROC-AUC: 0.9599474004570878


In [30]:
!pip freeze > requirements.txt